In [0]:
--ANALYSIS
--1. Which restaurants have the highest average overall rating, with at least 5 ratings?
SELECT rest.name, ROUND(AVG(r.overall_rating), 2) AS avg_rating, COUNT(*) AS num_ratings
FROM restaurants rest
JOIN ratings r
ON r.restaurant_id = rest.restaurant_id
GROUP BY rest.restaurant_id, rest.name
HAVING COUNT(*) >= 5
ORDER BY AVG(r.overall_rating) DESC;


--2. What are the most preferred cuisines among consumers?
SELECT preferred_cuisine, COUNT(*) AS fav_cuisine
FROM consumer_preferences
GROUP BY preferred_cuisine
ORDER BY COUNT(*) DESC
LIMIT 10;

-- 3. Which cuisines have the highest average restaurant rating?
SELECT rest.cuisine, ROUND(AVG(r.overall_rating), 2), COUNT(*) AS num_ratings
FROM restaurant_cuisines rest
JOIN ratings r
ON r.restaurant_id = rest.restaurant_id
GROUP BY rest.cuisine
ORDER BY ROUND(AVG(r.overall_rating), 2) DESC;

-- 4. Do expensive restaurants (Price = High) get rated higher than cheap ones?
SELECT rest.price, ROUND(AVG(r.overall_rating), 2) AS avg_rating, COUNT(*) AS num_ratings
FROM restaurants rest
JOIN ratings r
ON r.restaurant_id = rest.restaurant_id
GROUP BY rest.price
ORDER BY avg_rating DESC;

--5. Which consumers are the harshest raters?
SELECT c.consumer_id, ROUND(AVG(r.overall_rating), 2) AS avg_rating, COUNT(*) AS num_ratings
FROM consumers c
JOIN ratings r ON c.consumer_id = r.consumer_id
GROUP BY c.consumer_id
HAVING COUNT(*) >= 5
ORDER BY avg_rating ASC;

-- 6. How does the average restaurant rating differ by city?
SELECT rest.city, ROUND(AVG(r.overall_rating), 2) AS avg_rating, COUNT(*) AS num_ratings
FROM restaurants rest
JOIN ratings r
ON r.restaurant_id = rest.restaurant_id
GROUP BY rest.city
HAVING COUNT(*) >= 5

-- 7. Cities with above-average restaurant ratings
WITH city_avg AS (
	SELECT rest.city, ROUND(AVG(r.overall_rating), 2) AS avg_rating
	FROM restaurantS rest
	JOIN ratings r
	ON rest.restaurant_id = r.restaurant_id
	GROUP BY rest.city
),
overall_avg AS (
	SELECT ROUND(AVG(overall_rating), 2) AS overall_avg_rating
	FROM ratings
)
SELECT ca.city, ca.avg_rating AS city_avg_rating, oa.overall_avg_rating
FROM city_avg AS ca
CROSS JOIN overall_avg AS oa
WHERE ca.avg_rating > oa.overall_avg_rating
ORDER BY ca.avg_rating DESC;

-- 8. Which consumers give ratings that are, on average, 
--    higher than the average rating given by consumers in their own city?
WITH consumer_avg AS (
	SELECT c.consumer_id, c.city, ROUND(AVG(r.overall_rating), 2) AS avg_consumer_rating
	FROM consumers c
	JOIN ratings r
	ON r.consumer_id = c.consumer_id
	GROUP BY c.consumer_id, c.city
	ORDER BY ROUND(AVG(r.overall_rating), 2)
),
city_avg AS (
	SELECT c.city, ROUND(AVG(r.overall_rating), 2) AS avg_city_rating
	FROM consumers c
	JOIN ratings r
	ON c.consumer_id = r.consumer_id
	GROUP BY c.city
	ORDER BY avg_city_rating
)
SELECT coa.consumer_id, coa.avg_consumer_rating, ca.avg_city_rating
FROM consumer_avg coa
JOIN city_avg ca
ON ca.city = coa.city
WHERE coa.avg_consumer_rating > ca.avg_city_rating;

-- 9. Which restaurants have an average overall rating higher than the 
-- average rating for their cuisine?
WITH rest_avg AS (
	SELECT rest.restaurant_id, rest.name, rc.cuisine, ROUND(AVG(r.overall_rating), 2) AS avg_rest_rating
	FROM restaurants rest
	JOIN ratings r
	ON r.restaurant_id = rest.restaurant_id
	JOIN restaurant_cuisines rc
	ON rc.restaurant_id = r.restaurant_id
	GROUP BY rest.restaurant_id, rc.cuisine
	ORDER BY avg_rest_rating
),
cuisine_avg AS (
	SELECT rc.cuisine, ROUND(AVG(r.overall_rating), 2) AS avg_cuisine_rating
	FROM restaurant_cuisines rc
	JOIN ratings r
	ON rc.restaurant_id = r.restaurant_id
	GROUP BY rc.cuisine
	ORDER BY avg_cuisine_rating
)
SELECT ra.restaurant_id, ra.name, ra.cuisine, ra.avg_rest_rating, ca.avg_cuisine_rating
FROM rest_avg ra
JOIN cuisine_avg ca
ON ca.cuisine = ra.cuisine
WHERE avg_rest_rating > avg_cuisine_rating; 

-- 10. For each city, which restaurant has the highest average overall rating (with at least 5 ratings)?
WITH avg_rest AS(
	SELECT rest.restaurant_id, rest.name, rest.city, ROUND(AVG(r.overall_rating), 2) AS avg_rating
	FROM restaurants rest
	JOIN ratings r
	ON rest.restaurant_id = r.restaurant_id
	GROUP BY rest.restaurant_id, rest.name, rest.city
	HAVING COUNT(*) >= 5
),
ranked AS (
	SELECT restaurant_id, name, city, avg_rating, ROW_NUMBER() OVER (PARTITION BY city ORDER BY avg_rating DESC) AS rn
	FROM avg_rest
)
SELECT *
FROM ranked 
WHERE rn = 1;

-- 11. Which cuisines are served by more restaurants than the number of consumers who prefer them?
WITH cuisine_supply AS (
	SELECT cuisine, COUNT(*) AS num_restaurants
	FROM restaurant_cuisines
	GROUP BY cuisine
),
cuisine_demand AS (
	SELECT preferred_cuisine, COUNT(*) AS num_consumers
	FROM consumer_preferences
	GROUP BY preferred_cuisine
)
SELECT cs.cuisine, cs.num_restaurants, cd.num_consumers
FROM cuisine_supply cs
FULL OUTER JOIN cuisine_demand cD
ON cs.cuisine = cd.preferred_cuisine
WHERE num_restaurants > num_consumers;

-- 12. Which consumers prefer more cuisines than the average consumer?
WITH consumer_prefer AS(
	SELECT consumer_id, COUNT(*) as num_cuisines
	FROM consumer_preferences
	GROUP BY consumer_id
),
average_consumer AS(
	SELECT AVG(num_cuisines) AS avg_cuisines
	FROM consumer_prefer
)
SELECT consumer_id, num_cuisines
FROM consumer_prefer
WHERE num_cuisines > (SELECT avg_cuisines FROM average_consumer)
ORDER BY num_cuisines DESC;

-- 13. What is the distribution of consumers by city and state?
SELECT city, state, COUNT(*) AS num_consumers
FROM consumers
GROUP BY city, state
ORDER BY num_consumers DESC;

WITH consumer_ages AS (
    SELECT
        CASE
            WHEN c.age BETWEEN 18 AND 25 THEN '18-25'
            WHEN c.age BETWEEN 26 AND 40 THEN '26-40'
            WHEN c.age BETWEEN 41 AND 60 THEN '41-60'
            WHEN c.age > 60               THEN '60+'
            ELSE 'Unknown'
        END AS age_bucket,
        cp.preferred_cuisine,
        COUNT(*) AS num_consumers
    FROM consumers c
    JOIN consumer_preferences cp
        ON cp.consumer_id = c.consumer_id
    WHERE c.age IS NOT NULL
    GROUP BY age_bucket, cp.preferred_cuisine
),
ranked AS (
    SELECT
        age_bucket,
        preferred_cuisine,
        num_consumers,
        ROW_NUMBER() OVER (
            PARTITION BY age_bucket
            ORDER BY num_consumers DESC
        ) AS rn
    FROM consumer_ages
)
SELECT
    age_bucket,
    preferred_cuisine,
    num_consumers,
    rn AS rank
FROM ranked
WHERE rn <= 3
ORDER BY age_bucket, rn;

SELECT city, smoking_allowed
FROM restaurants
WHERE smoking_allowed IN ('Yes', 'No')
GROUP BY city, smoking_allowed;

SELECT rest.alcohol_service, 
	CASE
        WHEN r.overall_rating < 0.67 THEN 'Unsatisfactory'
        WHEN r.overall_rating < 1.34 THEN 'Satisfactory'
        ELSE                            'Highly Satisfactory'
    END AS rating_band,
	COUNT(*) AS num_ratings
FROM restaurants rest
JOIN ratings r
ON r.restaurant_id = rest.restaurant_id
GROUP BY rest.alcohol_service, rating_band
ORDER BY rest.alcohol_service;